# Grammar-only LoRA fine-tuning — Llama-3.1-8B

**Goal:** teach the rigid `ASSUME:/ASK:` grammar by plain next-token prediction on raw RG text, then measure story/literal/two-stage translation on the frozen eval_v1 (SAIR-derived, 777 problems).

**Dataset:** `train_v1/` (2,772 train + 100 holdout rows; ETP + genform laws, pair-disjoint from all SAIR, law-disjoint from eval_v1 — see `train_v1/manifest.json`). Eval source: [SAIRfoundation/equational-theories-selected-problems](https://huggingface.co/datasets/SAIRfoundation/equational-theories-selected-problems).

**Source attribution:** all logic lives in `training/train_lora.py` (this notebook never forks it); repo layout conventions after Shivam's vlm-alignment repo; renderers/grader are Oren's `informalizing-etp/`.

**Sweep convention:** Phase 5b varies rank ONLY over `RANKS = [1, 2, 8, 16, 32, 64]`, single layer `LAYER = 16` (`o_proj`), `lora_alpha == rank` always, seeds 0/1/2 on headline configs. Everything else (LR 2e-4 cosine, batch 4, seq 1024 packed, data) is frozen in `training/config.py`.

In [ ]:
# CONFIG — just change the rank here (and layer/seed if sweeping)
RANK = 16
LAYER = -1      # -1 = all layers (Phase 5a); 16 = single-layer constrained runs
SEED = 0
PRESET = ""    # or "smoke" / "phase5a" to reproduce the committed runs exactly

In [ ]:
# Launch training on Modal (same entry point as the CLI — no logic fork).
import subprocess, sys
cmd = ["modal", "run", "train_lora.py"]
cmd += (["--preset", PRESET] if PRESET else ["--rank", str(RANK), "--layer", str(LAYER), "--seed", str(SEED)])
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## Inference smoke test
5-problem story-arm eval of the adapter you just trained (full runbook: `eval/README.md`).

In [ ]:
RUN_NAME = PRESET and {"smoke": "smoke-r1-L16-oproj", "phase5a": "phase5a-r16-all"}[PRESET] or f"r{RANK}-l{'all' if LAYER == -1 else LAYER}-s{SEED}"
subprocess.run(["bash", "../eval/run_eval.sh", "8b", "story", "--limit", "5",
                "--adapter", f"/models/checkpoints/{RUN_NAME}/final", "--adapter-rank", str(RANK)], check=True)